# EDA completo: todas las variables y validación de datos faltantes

Este notebook perfila cada columna de los archivos y revisa observaciones nulas, fechas inválidas (`NaT`), fechas diarias ocultas, cobertura por estación-año, duplicados, ceros, valores extremos y disponibilidad del satélite. Sigue las ideas de las lecciones 4 y 5.

No corrige ni elimina datos. Los ceros se consideran valores observados hasta comprobar con los metadatos que sean códigos de faltantes. Para saber si una fecha ausente debía tener medición se requiere el historial operativo de cada estación.

In [7]:
!pip install numpy pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [4]:
!pip install --upgrade pip



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 5.9 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1


## 1. Preparación y localización de los archivos

El Excel es grande; leerlo puede tardar. El bloque busca los datos junto al notebook o en la carpeta de trabajo del proyecto.

In [5]:
from pathlib import Path
import zipfile
from collections import Counter
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    PLOT_AVAILABLE = True
except ImportError:
    PLOT_AVAILABLE = False
    print("matplotlib no instalado: se omitirán las gráficas, pero las tablas y auditorías sí se ejecutan.")

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 30)

# Si el notebook está en otra carpeta, cambia esta ruta a la del proyecto.
ROOT = Path.cwd()
required = ['P_1980_1990.csv', 'P_1991_2003.csv', 'P_1930_1979.xlsx', 'ImagenesSatelitePentadal.zip']
if not all((ROOT / f).exists() for f in required):
    candidates = [Path('/Users/usuario/Documents/gustavo/Analitica/proyecto3'), Path.cwd()]
    ROOT = next((p for p in candidates if all((p / f).exists() for f in required)), ROOT)
print('Carpeta de datos:', ROOT)
print('Archivos encontrados:')
for f in required:
    p = ROOT / f
    print(f' - {f}: {p.stat().st_size / 1e6:.2f} MB' if p.exists() else f' - FALTA: {f}')

matplotlib no instalado: se omitirán las gráficas, pero las tablas y auditorías sí se ejecutan.
Carpeta de datos: /Users/usuario/Documents/gustavo/Analitica/proyecto3
Archivos encontrados:
 - P_1980_1990.csv: 24.12 MB
 - P_1991_2003.csv: 24.42 MB
 - P_1930_1979.xlsx: 28.38 MB
 - ImagenesSatelitePentadal.zip: 6.75 MB


## 2. Carga y forma de las tablas de estaciones

`VALUE` es el valor de precipitación registrado, pero los archivos no traen aquí una unidad visible para los CSV. No asumimos milímetros hasta confirmarlo con la documentación. El Excel incluye una columna `UNITS` que revisaremos aparte.

In [8]:
csv_files = {
    'csv_1980_1990': ROOT / 'P_1980_1990.csv',
    'csv_1991_2003': ROOT / 'P_1991_2003.csv',
}
frames = {}
for source, path in csv_files.items():
    frames[source] = pd.read_csv(path, dtype={'STATION_ID': 'string'})
    print('\n', source, frames[source].shape)
    display(frames[source].head())

# El histórico se carga por separado para no mezclar periodos ni fuentes.
hist = pd.read_excel(ROOT / 'P_1930_1979.xlsx', sheet_name='Hoja1', dtype={'STATION_ID': 'string'})
print('Excel histórico:', hist.shape)
display(hist.head())
print('Columnas CSV:', list(frames['csv_1980_1990'].columns))
print('Columnas Excel:', list(hist.columns))
if 'UNITS' in hist.columns:
    display(hist['UNITS'].value_counts(dropna=False))


 csv_1980_1990 (997791, 5)


,STATION_ID,YEAR,MONTH,DAY,VALUE
0,2610000102,1980,1,1,2.0
1,2610000103,1980,1,1,5.0
2,2610000104,1980,1,1,8.0
3,2610000105,1980,1,1,7.0
4,2610000107,1980,1,1,0.0



 csv_1991_2003 (1010677, 5)


,STATION_ID,YEAR,MONTH,DAY,VALUE
0,2610000101,1991,1,1,0.0
1,2610000102,1991,1,1,0.0
2,2610000103,1991,1,1,6.0
3,2610000104,1991,1,1,0.0
4,2610000105,1991,1,1,0.0


Excel histórico: (984226, 7)


,STATION_ID,SENSORNAME,YEAR,MONTH,DAY,VALUE,UNITS
0,2610000101,Pluviometer,1952,1,18,0.0,mm
1,2610000101,Pluviometer,1952,1,19,0.0,mm
2,2610000101,Pluviometer,1952,1,20,0.0,mm
3,2610000101,Pluviometer,1952,1,21,0.0,mm
4,2610000101,Pluviometer,1952,1,22,8.0,mm


Columnas CSV: ['STATION_ID', 'YEAR', 'MONTH', 'DAY', 'VALUE']
Columnas Excel: ['STATION_ID', 'SENSORNAME', 'YEAR', 'MONTH', 'DAY', 'VALUE', 'UNITS']


UNITS
mm    984226
Name: count, dtype: int64

In [19]:
frames


{'csv_1980_1990':         STATION_ID  YEAR  MONTH  DAY  VALUE
 0       2610000102  1980      1    1    2.0
 1       2610000103  1980      1    1    5.0
 2       2610000104  1980      1    1    8.0
 3       2610000105  1980      1    1    7.0
 4       2610000107  1980      1    1    0.0
 ...            ...   ...    ...  ...    ...
 997786  2610000113  1990     12   31   30.0
 997787  5300000201  1990     12   31   31.0
 997788  5300000202  1990     12   31  110.0
 997789  5421200201  1990     12   31   66.0
 997790  5400000201  1990     12   31    0.0
 
 [997791 rows x 5 columns],
 'csv_1991_2003':          STATION_ID  YEAR  MONTH  DAY  VALUE
 0        2610000101  1991      1    1    0.0
 1        2610000102  1991      1    1    0.0
 2        2610000103  1991      1    1    6.0
 3        2610000104  1991      1    1    0.0
 4        2610000105  1991      1    1    0.0
 ...             ...   ...    ...  ...    ...
 1010672  5400000201  2003     12   31    8.0
 1010673  2612000113  2003  

## 3. Nulos explícitos y problemas de formato

Un nulo explícito es una celda vacía o un marcador que pandas reconoce como nulo. Esto es distinto a una fecha que no aparece en absoluto. Revisamos además si fechas o mediciones no se pueden convertir a números/fechas válidos.

In [9]:
def null_audit(df, name):
    out = pd.DataFrame({
        "columna": df.columns,
        "nulos": [int(df[c].isna().sum()) for c in df.columns],
        "porcentaje_nulos": [float(df[c].isna().mean() * 100) for c in df.columns],
        "tipo": [str(df[c].dtype) for c in df.columns],
    })
    out.insert(0, "fuente", name)
    return out

nulls = pd.concat([null_audit(df, name) for name, df in frames.items()] + [null_audit(hist, "excel_1930_1979")], ignore_index=True)
display(nulls)

def prepare_station(df, source):
    x = df.copy()
    x["source"] = source
    for col in ["YEAR", "MONTH", "DAY", "VALUE"]:
        x[col + "_num"] = pd.to_numeric(x[col], errors="coerce")
    x["DATE"] = pd.to_datetime(dict(year=x.YEAR_num, month=x.MONTH_num, day=x.DAY_num), errors="coerce")
    return x

station = pd.concat([prepare_station(df, name) for name, df in frames.items()], ignore_index=True)
history = prepare_station(hist, "excel_1930_1979")
format_audit = pd.DataFrame([
    {"fuente": name, "filas": len(df), "id_nulo": int(df.STATION_ID.isna().sum()),
     "valor_no_numerico": int(pd.to_numeric(df.VALUE, errors="coerce").isna().sum()),
     "fecha_invalida": int(prepare_station(df, name).DATE.isna().sum())}
    for name, df in list(frames.items()) + [("excel_1930_1979", hist)]
])
display(format_audit)


,fuente,columna,nulos,porcentaje_nulos,tipo
0,csv_1980_1990,STATION_ID,0,0.0,string
1,csv_1980_1990,YEAR,0,0.0,int64
2,csv_1980_1990,MONTH,0,0.0,int64
3,csv_1980_1990,DAY,0,0.0,int64
4,csv_1980_1990,VALUE,0,0.0,float64
5,csv_1991_2003,STATION_ID,0,0.0,string
6,csv_1991_2003,YEAR,0,0.0,int64
7,csv_1991_2003,MONTH,0,0.0,int64
8,csv_1991_2003,DAY,0,0.0,int64
9,csv_1991_2003,VALUE,0,0.0,float64


,fuente,filas,id_nulo,valor_no_numerico,fecha_invalida
0,csv_1980_1990,997791,0,0,0
1,csv_1991_2003,1010677,0,0,0
2,excel_1930_1979,984226,0,0,0


## Perfil de todas las variables

Este perfil se aplica a cada columna del CSV y del Excel: tipo detectado, nulos explícitos, cantidad de valores distintos y ejemplos más frecuentes. Para variables numéricas agrega rango y percentiles. Es un resumen para orientar la revisión; los histogramas y tablas de frecuencia ayudan a reconocer códigos, concentraciones y valores raros.


In [10]:
source_frames = {**frames, "excel_1930_1979": hist}
profile_rows = []
for source, df in source_frames.items():
    for col in df.columns:
        s = df[col]
        nonnull = s.dropna()
        row = {"source": source, "variable": col, "dtype": str(s.dtype),
               "rows": len(s), "nulls": int(s.isna().sum()),
               "null_pct": float(s.isna().mean() * 100), "distinct_nonnull": int(nonnull.nunique())}
        numeric = pd.to_numeric(s, errors="coerce")
        if col not in ["STATION_ID", "SENSORNAME", "UNITS"] and numeric.notna().sum() and numeric.notna().sum() >= max(5, int(.8 * nonnull.shape[0])):
            row.update({"numeric_min": numeric.min(), "numeric_p01": numeric.quantile(.01),
                        "numeric_median": numeric.median(), "numeric_mean": numeric.mean(),
                        "numeric_p99": numeric.quantile(.99), "numeric_max": numeric.max()})
        else:
            row["most_common_values"] = nonnull.astype(str).value_counts().head(8).to_dict()
        profile_rows.append(row)
variable_profile = pd.DataFrame(profile_rows)
display(variable_profile)

# Frecuencias de las variables discretas de calendario y de metadatos de estación.
for source, df in source_frames.items():
    print("\nFrecuencias útiles para", source)
    for col in ["YEAR", "MONTH", "DAY", "SENSORNAME", "UNITS"]:
        if col in df.columns:
            print("\n", col, "(frecuencias más comunes)")
            display(df[col].value_counts(dropna=False).head(20).rename_axis(col).reset_index(name="count"))
    if "STATION_ID" in df.columns:
        print("\nEstaciones con más registros")
        display(df.STATION_ID.value_counts(dropna=False).head(15).rename_axis("STATION_ID").reset_index(name="rows"))

if PLOT_AVAILABLE:
    # Comparación gráfica de VALUE. Un panel por archivo evita mezclar épocas.
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (source, df) in zip(axes, source_frames.items()):
        v = pd.to_numeric(df.VALUE, errors="coerce").dropna()
        positive = v[v > 0]
        ax.hist(positive.clip(upper=positive.quantile(.99)), bins=45, color="teal", alpha=.85)
        ax.set_title(source); ax.set_xlabel("VALUE positivo, limitado visualmente al P99"); ax.set_ylabel("Frecuencia")
        ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()


,source,variable,dtype,rows,nulls,null_pct,distinct_nonnull,most_common_values,numeric_min,numeric_p01,numeric_median,numeric_mean,numeric_p99,numeric_max
0,csv_1980_1990,STATION_ID,string,997791,0,0.0,302,"{'2620610103': 4018, '2612200103': 4018, '2612...",NaN,NaN,NaN,NaN,NaN,NaN
1,csv_1980_1990,YEAR,int64,997791,0,0.0,11,NaN,1980.0,1980.0,1985.0,1985.150454,1990.0,1990.0
2,csv_1980_1990,MONTH,int64,997791,0,0.0,12,NaN,1.0,1.0,7.0,6.530437,12.0,12.0
3,csv_1980_1990,DAY,int64,997791,0,0.0,31,NaN,1.0,1.0,16.0,15.736650,31.0,31.0
4,csv_1980_1990,VALUE,float64,997791,0,0.0,967,NaN,0.0,0.0,0.0,6.611451,60.0,859375.0
5,csv_1991_2003,STATION_ID,string,1010677,0,0.0,284,"{'2612000109': 4748, '2612000105': 4748, '2612...",NaN,NaN,NaN,NaN,NaN,NaN
6,csv_1991_2003,YEAR,int64,1010677,0,0.0,13,NaN,1991.0,1991.0,1996.0,1996.645689,2003.0,2003.0
7,csv_1991_2003,MONTH,int64,1010677,0,0.0,12,NaN,1.0,1.0,7.0,6.496432,12.0,12.0
8,csv_1991_2003,DAY,int64,1010677,0,0.0,31,NaN,1.0,1.0,16.0,15.727002,31.0,31.0
9,csv_1991_2003,VALUE,float64,1010677,0,0.0,1151,NaN,0.0,0.0,0.0,5.469390,60.0,600.0



Frecuencias útiles para csv_1980_1990

 YEAR (frecuencias más comunes)


,YEAR,count
0,1990,96211
1,1987,94679
2,1986,93398
3,1985,93218
4,1988,93108
5,1989,93091
6,1984,92879
7,1983,91157
8,1982,87873
9,1981,84320



 MONTH (frecuencias más comunes)


,MONTH,count
0,8,85162
1,7,84906
2,10,84798
3,3,84694
4,12,84493
5,5,84433
6,1,83969
7,6,82243
8,9,82171
9,11,82148



 DAY (frecuencias más comunes)


... (19 outputs left out)


 UNITS (frecuencias más comunes)


,UNITS,count
0,mm,984226



Estaciones con más registros


,STATION_ID,rows
0,2612700202,18255
1,2612700201,16994
2,2620500201,13214
3,2622400101,12579
4,2611200101,11948
5,2611400101,11927
6,2622100301,11921
7,2622100201,11855
8,2620000101,11694
9,2620610101,11527


## Verificación detallada de datos faltantes: observaciones y fechas

Siguiendo las lecciones, separamos:

- **Nulos explícitos:** una celda vacía en `VALUE` u otro campo.
- **Fechas inválidas (`NaT`):** falta algún componente o la combinación año-mes-día no existe.
- **Fechas ocultas:** el registro con fecha no está en la tabla, aunque la secuencia debería ser diaria. Se detecta por estación comparando las fechas observadas con `date_range(...).difference(...)`.
- **Cobertura calendario:** comparación de fechas observadas con todos los días del año. Los primeros o últimos días ausentes pueden estar fuera del periodo operativo; por eso también se calcula la falta interna entre primera y última observación.

No se rellenan las fechas faltantes con lluvia cero: reindexar sirve para hacer visible el hueco, pero el valor correcto sigue siendo desconocido.


In [11]:
# Nulos en las columnas originales antes de convertir tipos.
null_rows = []
for source, df in source_frames.items():
    for col in df.columns:
        null_rows.append({"source": source, "variable": col,
                          "null_count": int(df[col].isna().sum()),
                          "null_pct": float(df[col].isna().mean() * 100)})
null_audit_all = pd.DataFrame(null_rows)
print("Nulos explícitos por variable")
display(null_audit_all)

# Fechas inválidas / valores que no se pueden interpretar como número.
prepared_all = {source: prepare_station(df, source) for source, df in source_frames.items()}
validity_rows = []
for source, x in prepared_all.items():
    validity_rows.append({"source": source, "rows": len(x),
                          "NaT_or_invalid_date": int(x.DATE.isna().sum()),
                          "VALUE_missing_or_not_numeric": int(x.VALUE_num.isna().sum()),
                          "negative_VALUE": int((x.VALUE_num < 0).sum())})
validity_audit = pd.DataFrame(validity_rows)
print("Fechas inválidas, valores faltantes y negativos")
display(validity_audit)

# Auditoría por estación, no mezclamos estaciones en una sola serie de tiempo.
hidden_gap_rows, station_year_rows, missing_station_year_rows = [], [], []
for source, x in prepared_all.items():
    valid = x.dropna(subset=["STATION_ID", "DATE"]).copy()
    valid["STATION_ID"] = valid.STATION_ID.astype(str)
    for station_id, g in valid.groupby("STATION_ID"):
        dates = pd.DatetimeIndex(g.DATE.drop_duplicates()).sort_values()
        if not len(dates):
            continue
        ideal_span = pd.date_range(dates.min(), dates.max(), freq="D")
        hidden_dates = ideal_span.difference(dates)
        hidden_gap_rows.append({"source": source, "STATION_ID": station_id,
                                "first_date": dates.min(), "last_date": dates.max(),
                                "unique_dates": len(dates), "hidden_dates_inside_span": len(hidden_dates),
                                "first_20_hidden_dates": list(hidden_dates[:20].strftime("%Y-%m-%d"))})
        years_seen = set(g.DATE.dt.year.astype(int))
        for year in range(min(years_seen), max(years_seen) + 1):
            year_dates = pd.DatetimeIndex(g.loc[g.DATE.dt.year == year, "DATE"].drop_duplicates()).sort_values()
            calendar = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
            missing_calendar = calendar.difference(year_dates)
            if len(year_dates):
                span = pd.date_range(year_dates.min(), year_dates.max(), freq="D")
                internal = span.difference(year_dates)
                first_seen, last_seen = year_dates.min(), year_dates.max()
                leading = int((first_seen - calendar[0]).days)
                trailing = int((calendar[-1] - last_seen).days)
            else:
                internal = pd.DatetimeIndex([])
                first_seen = last_seen = pd.NaT
                leading, trailing = len(calendar), 0
                missing_station_year_rows.append({"source": source, "STATION_ID": station_id, "YEAR": year})
            station_year_rows.append({"source": source, "STATION_ID": station_id, "YEAR": year,
                                      "expected_days": len(calendar), "unique_dates": len(year_dates),
                                      "missing_vs_full_calendar": len(missing_calendar),
                                      "missing_inside_first_last": len(internal),
                                      "leading_days": leading, "trailing_days": trailing,
                                      "coverage_pct": 100 * len(year_dates) / len(calendar),
                                      "first_record": first_seen, "last_record": last_seen})

hidden_gaps = pd.DataFrame(hidden_gap_rows)
station_year_detail = pd.DataFrame(station_year_rows)
print("Resumen de huecos ocultos en la serie diaria de cada estación")
display(hidden_gaps.groupby("source").agg(
    stations=("STATION_ID", "nunique"),
    stations_with_hidden_days=("hidden_dates_inside_span", lambda s: int((s > 0).sum())),
    total_hidden_days_inside_span=("hidden_dates_inside_span", "sum"),
    max_hidden_days_for_station=("hidden_dates_inside_span", "max")).reset_index())

# Duplicados de la llave de medición: el mismo ID y la misma fecha.
duplicate_rows = []
for source, x in prepared_all.items():
    valid = x.dropna(subset=["STATION_ID", "DATE"]).copy()
    counts = valid.groupby(["STATION_ID", "DATE"]).size()
    duplicate_rows.append({"source": source, "duplicate_station_date_keys": int((counts > 1).sum()),
                           "extra_rows_in_duplicate_keys": int((counts[counts > 1] - 1).sum().sum())})
print("Duplicados de estación-fecha")
display(pd.DataFrame(duplicate_rows))

print("Pares estación-año con menor cobertura del calendario; se incluyen extremos posiblemente fuera de operación")
display(station_year_detail.sort_values("coverage_pct").head(25))
print("Años de una estación sin ninguna fila entre su primer y último año observado")
display(pd.DataFrame(missing_station_year_rows).head(30))

# Muestra de diferencia entre calendario ideal y fechas reales para revisar un caso.
worst = station_year_detail.sort_values("coverage_pct").iloc[0]
raw = prepared_all[worst.source]
observed = pd.DatetimeIndex(raw.loc[(raw.STATION_ID.astype(str) == str(worst.STATION_ID)) &
                                    (raw.DATE.dt.year == int(worst.YEAR)), "DATE"].dropna().drop_duplicates()).sort_values()
calendar = pd.date_range(f"{int(worst.YEAR)}-01-01", f"{int(worst.YEAR)}-12-31", freq="D")
missing_dates_example = calendar.difference(observed)
print("Fechas faltantes del par estación-año de menor cobertura:", worst.source, worst.STATION_ID, int(worst.YEAR),
      "total=", len(missing_dates_example))
display(pd.DataFrame({"missing_date": missing_dates_example[:50]}))

# Guarda las tablas de auditoría para poder rastrear hallazgos sin cambiar la data cruda.
OUTPUT = ROOT / "eda_resultados"
OUTPUT.mkdir(exist_ok=True)
variable_profile.to_csv(OUTPUT / "perfil_todas_las_variables.csv", index=False)
null_audit_all.to_csv(OUTPUT / "nulos_por_variable.csv", index=False)
validity_audit.to_csv(OUTPUT / "validez_fechas_y_valores.csv", index=False)
hidden_gaps.to_csv(OUTPUT / "huecos_diarios_por_estacion.csv", index=False)
station_year_detail.to_csv(OUTPUT / "auditoria_completa_estacion_anio.csv", index=False)
print("Tablas de perfil y faltantes guardadas en", OUTPUT)


Nulos explícitos por variable


,source,variable,null_count,null_pct
0,csv_1980_1990,STATION_ID,0,0.0
1,csv_1980_1990,YEAR,0,0.0
2,csv_1980_1990,MONTH,0,0.0
3,csv_1980_1990,DAY,0,0.0
4,csv_1980_1990,VALUE,0,0.0
5,csv_1991_2003,STATION_ID,0,0.0
6,csv_1991_2003,YEAR,0,0.0
7,csv_1991_2003,MONTH,0,0.0
8,csv_1991_2003,DAY,0,0.0
9,csv_1991_2003,VALUE,0,0.0


Fechas inválidas, valores faltantes y negativos


,source,rows,NaT_or_invalid_date,VALUE_missing_or_not_numeric,negative_VALUE
0,csv_1980_1990,997791,0,0,0
1,csv_1991_2003,1010677,0,0,0
2,excel_1930_1979,984226,0,0,0


Resumen de huecos ocultos en la serie diaria de cada estación


,source,stations,stations_with_hidden_days,total_hidden_days_inside_span,max_hidden_days_for_station
0,csv_1980_1990,302,236,34273,1822
1,csv_1991_2003,284,223,36367,2098
2,excel_1930_1979,264,220,73617,8694


... (5 outputs left out)

,source,STATION_ID,YEAR
0,csv_1980_1990,2612700202,1988
1,csv_1980_1990,2612700202,1989
2,csv_1980_1990,2612710107,1985
3,csv_1980_1990,2612710201,1988
4,csv_1980_1990,2612710201,1989
5,csv_1980_1990,2612800107,1983
6,csv_1980_1990,2612810121,1981
7,csv_1980_1990,2612810121,1982
8,csv_1980_1990,2614500201,1988
9,csv_1980_1990,5400000201,1983


Fechas faltantes del par estación-año de menor cobertura: csv_1980_1990 5421200201 1986 total= 365


,missing_date
0,1986-01-01
1,1986-01-02
2,1986-01-03
3,1986-01-04
4,1986-01-05
...,...
45,1986-02-15
46,1986-02-16
47,1986-02-17
48,1986-02-18


Tablas de perfil y faltantes guardadas en /Users/usuario/Documents/gustavo/Analitica/proyecto3/eda_resultados


## 4. ¿Qué tan completos son los registros diarios?

Para cada estación-año contamos filas, fechas únicas y cobertura del calendario. Los días faltantes se calculan como días del año menos fechas únicas observadas. Esto identifica huecos implícitos; no determina si la falta es accidental ni si un cero es dato faltante.

In [12]:
def make_coverage(df):
    x = df.dropna(subset=["STATION_ID", "DATE", "VALUE_num"]).copy()
    x["YEAR_INT"] = x.DATE.dt.year
    cov = x.groupby(["source", "STATION_ID", "YEAR_INT"], as_index=False).agg(
        rows=("VALUE_num", "size"), unique_dates=("DATE", "nunique"),
        zero_count=("VALUE_num", lambda s: int((s == 0).sum())),
        annual_sum_observed=("VALUE_num", "sum"),
        first_date=("DATE", "min"), last_date=("DATE", "max"))
    cov["expected_days"] = cov.YEAR_INT.map(lambda y: 366 if pd.Timestamp(y, 12, 31).dayofyear == 366 else 365)
    cov["missing_calendar_days"] = cov.expected_days - cov.unique_dates
    cov["coverage_pct"] = 100 * cov.unique_dates / cov.expected_days
    return cov

coverage = pd.concat([make_coverage(station), make_coverage(history)], ignore_index=True)
annual_presence = coverage.groupby(["source", "YEAR_INT"]).agg(
    stations=("STATION_ID", "nunique"), station_year_pairs=("STATION_ID", "size"),
    median_coverage_pct=("coverage_pct", "median"),
    min_coverage_pct=("coverage_pct", "min"),
    pairs_below_90pct=("coverage_pct", lambda s: int((s < 90).sum()))).reset_index()
print("Resumen por año: estaciones con algún registro, cobertura y pares bajo 90%")
display(annual_presence)
for source, group in coverage.groupby("source"):
    years = set(group.YEAR_INT.astype(int))
    missing_years = sorted(set(range(min(years), max(years) + 1)) - years)
    print(source, "años con algún dato:", min(years), "a", max(years), "años vacíos dentro del rango:", missing_years)
print("Atención: la presencia de al menos una estación no significa que todas las estaciones tengan datos ese año.")
print("Estaciones-año con menos de 90% cobertura:", int((coverage.coverage_pct < 90).sum()))
display(coverage.sort_values("coverage_pct").head(15))

if PLOT_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 4))
    for source, group in coverage.groupby("source"):
        ax.hist(group.coverage_pct, bins=np.arange(0, 105, 5), alpha=.55, label=source)
    ax.set(xlabel="Cobertura de fechas del año (%)", ylabel="Número de pares estación-año", title="Cobertura anual de estaciones")
    ax.legend(); ax.grid(alpha=.2); plt.show()


Resumen por año: estaciones con algún registro, cobertura y pares bajo 90%


,source,YEAR_INT,stations,station_year_pairs,median_coverage_pct,min_coverage_pct,pairs_below_90pct
0,csv_1980_1990,1980,229,229,100.0,3.278689,34
1,csv_1980_1990,1981,245,245,100.0,3.561644,34
2,csv_1980_1990,1982,260,260,100.0,9.863014,46
3,csv_1980_1990,1983,261,261,100.0,8.493151,25
4,csv_1980_1990,1984,264,264,100.0,13.661202,20
...,...,...,...,...,...,...,...
69,excel_1930_1979,1975,232,232,100.0,8.493151,32
70,excel_1930_1979,1976,234,234,100.0,3.825137,24
71,excel_1930_1979,1977,235,235,100.0,28.767123,22
72,excel_1930_1979,1978,227,227,100.0,16.712329,27


csv_1980_1990 años con algún dato: 1980 a 1990 años vacíos dentro del rango: []
csv_1991_2003 años con algún dato: 1991 a 2003 años vacíos dentro del rango: []
excel_1930_1979 años con algún dato: 1930 a 1979 años vacíos dentro del rango: []
Atención: la presencia de al menos una estación no significa que todas las estaciones tengan datos ese año.
Estaciones-año con menos de 90% cobertura: 1042


,source,STATION_ID,YEAR_INT,rows,unique_dates,zero_count,annual_sum_observed,first_date,last_date,expected_days,missing_calendar_days,coverage_pct
7340,excel_1930_1979,2620000203,1979,1,1,1,0.0,1979-01-01,1979-01-01,365,364,0.273973
5657,csv_1991_2003,5421200104,1994,3,3,0,45.0,1994-09-07,1994-09-21,365,362,0.821918
6419,excel_1930_1979,2612710101,1971,7,7,0,1167.0,1971-01-31,1971-07-31,365,358,1.917808
6619,excel_1930_1979,2612810107,1969,10,10,9,11.0,1969-12-22,1969-12-31,365,355,2.739726
6296,excel_1930_1979,2612700201,1933,11,11,0,1359.0,1933-01-31,1933-12-31,365,354,3.013699
1528,csv_1980_1990,2620610101,1980,12,12,0,2391.0,1980-01-31,1980-12-31,366,354,3.278689
6418,excel_1930_1979,2612710101,1970,12,12,0,1101.0,1970-01-31,1970-12-31,365,353,3.287671
6914,excel_1930_1979,2613600301,1971,12,12,0,1298.0,1971-01-31,1971-12-31,365,353,3.287671
476,csv_1980_1990,2612200101,1985,12,12,9,31.0,1985-01-01,1985-12-01,365,353,3.287671
2758,csv_1980_1990,5421200112,1981,13,13,0,2951.0,1981-10-07,1981-12-26,365,352,3.561644


## 5. Duplicados y fechas repetidas

Una fecha repetida puede ser duplicación accidental o varias mediciones para un mismo día. Hay que resolverlo antes de construir curvas anuales. La búsqueda se hace por fuente, estación y fecha.

In [13]:
def find_duplicates(df):
    x = df.dropna(subset=['STATION_ID', 'DATE']).copy()
    counts = x.groupby(['source', 'STATION_ID', 'DATE']).size().rename('n_filas').reset_index()
    return counts[counts.n_filas > 1].sort_values('n_filas', ascending=False)

duplicates = find_duplicates(station)
print('Grupos de estación-fecha repetidos:', len(duplicates))
display(duplicates.head(20))

# Filas completamente idénticas (excluye las columnas auxiliares).
for name, df in frames.items():
    print(name, 'filas duplicadas idénticas:', int(df.duplicated().sum()))

Grupos de estación-fecha repetidos: 0


,source,STATION_ID,DATE,n_filas


csv_1980_1990 filas duplicadas idénticas: 0
csv_1991_2003 filas duplicadas idénticas: 0


## 6. Ceros, distribución y valores extremos

En lluvia, muchos ceros pueden ser días secos reales. Sin un código de calidad o manual de la fuente, no se pueden cambiar automáticamente a nulos. Los máximos extremos también pueden ser reales, acumulados por varios días o errores de digitación. Primero localizamos esos registros y los mostramos con estación y fecha.

In [14]:
def value_summary(df, source):
    v = pd.to_numeric(df.VALUE, errors='coerce')
    return {'fuente': source, 'n': int(v.notna().sum()), 'ceros': int((v == 0).sum()),
            'ceros_pct': 100 * float((v == 0).mean()), 'negativos': int((v < 0).sum()),
            'media': v.mean(), 'mediana': v.median(), 'p99': v.quantile(.99),
            'p999': v.quantile(.999), 'maximo': v.max(), 'mayores_500': int((v > 500).sum())}

value_stats = pd.DataFrame([value_summary(df, name) for name, df in frames.items()] + [value_summary(hist, 'excel_1930_1979')])
display(value_stats.round(3))

def show_extremes(df, source, threshold=100):
    x = prepare_station(df, source)
    return x.loc[x.VALUE_num >= threshold, ['STATION_ID','YEAR','MONTH','DAY','DATE','VALUE']].sort_values('VALUE', ascending=False)

for name, df in list(frames.items()) + [('excel_1930_1979', hist)]:
    print('Valores >= 100:', name)
    display(show_extremes(df, name).head(20))

# Histograma separado: el pico en cero tapa la forma de los valores positivos.
positive = station.loc[station.VALUE_num > 0, 'VALUE_num'].dropna()
if PLOT_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(positive.clip(upper=positive.quantile(.99)), bins=60, color='teal', edgecolor='white')
    ax.set(title='Valores positivos de lluvia (recortados visualmente al percentil 99)', xlabel='VALUE', ylabel='Frecuencia')
    ax.grid(alpha=.2); plt.show()
# Revisión exploratoria de valores grandes justo al cierre del mes.
# Esto ayuda a detectar posibles acumulados mensuales digitados en un solo día.
for name, df in list(frames.items()) + [("excel_1930_1979", hist)]:
    x = prepare_station(df, name).dropna(subset=["DATE", "VALUE_num"]).copy()
    x["es_ultimo_dia_mes"] = x.DATE.dt.is_month_end
    hi = x[x.VALUE_num >= 100]
    if len(hi):
        print(name, "valores >=100:", len(hi), "en fin de mes:", int(hi.es_ultimo_dia_mes.sum()),
              "porcentaje fin de mes:", round(100 * hi.es_ultimo_dia_mes.mean(), 1))
        display(hi.sort_values("VALUE_num", ascending=False)[["STATION_ID","DATE","VALUE","es_ultimo_dia_mes"]].head(15))


,fuente,n,ceros,ceros_pct,negativos,media,mediana,p99,p999,maximo,mayores_500
0,csv_1980_1990,997791,578166,57.945,0,6.611,0.0,60.0,126.0,859375.0,11
1,csv_1991_2003,1010677,597017,59.071,0,5.469,0.0,60.0,129.0,600.0,3
2,excel_1930_1979,984226,559657,56.863,0,5.427,0.0,55.0,104.0,736.0,14


Valores >= 100: csv_1980_1990


,STATION_ID,YEAR,MONTH,DAY,DATE,VALUE
343718,2611600101,1984,1,10,1984-01-10,859375.0
147876,5421200201,1981,10,31,1981-10-31,943.0
112351,5421200201,1981,5,31,1981-05-31,896.0
105202,5421200201,1981,4,30,1981-04-30,851.0
596988,5300000201,1986,9,30,1986-09-30,838.0
126496,5421200201,1981,7,31,1981-07-31,750.0
589279,5300000201,1986,8,31,1986-08-31,732.0
573342,5300000201,1986,6,30,1986-06-30,681.0
565607,5300000201,1986,5,31,1986-05-31,630.0
140734,5421200201,1981,9,30,1981-09-30,592.0


Valores >= 100: csv_1991_2003


,STATION_ID,YEAR,MONTH,DAY,DATE,VALUE
962516,2614400105,2003,4,9,2003-04-09,600.0
277937,5300000203,1994,3,31,1994-03-31,560.0
503242,2615400102,1996,12,8,1996-12-08,503.0
76261,5301400104,1991,11,9,1991-11-09,421.0
425655,5300900102,1995,12,19,1995-12-19,380.0
399657,5301500106,1995,8,31,1995-08-31,343.0
374240,5300000203,1995,5,15,1995-05-15,338.0
904382,5300000203,2002,5,24,2002-05-24,335.0
237700,5300000202,1993,10,8,1993-10-08,310.7
572486,5300000203,1997,10,27,1997-10-27,288.0


Valores >= 100: excel_1930_1979


... (3 outputs left out)

csv_1991_2003 valores >=100: 2473 en fin de mes: 108 porcentaje fin de mes: 4.4


,STATION_ID,DATE,VALUE,es_ultimo_dia_mes
962516,2614400105,2003-04-09,600.0,False
277937,5300000203,1994-03-31,560.0,True
503242,2615400102,1996-12-08,503.0,False
76261,5301400104,1991-11-09,421.0,False
425655,5300900102,1995-12-19,380.0,False
399657,5301500106,1995-08-31,343.0,True
374240,5300000203,1995-05-15,338.0,False
904382,5300000203,2002-05-24,335.0,False
237700,5300000202,1993-10-08,310.7,False
572486,5300000203,1997-10-27,288.0,False


excel_1930_1979 valores >=100: 1219 en fin de mes: 130 porcentaje fin de mes: 10.7


,STATION_ID,DATE,VALUE,es_ultimo_dia_mes
963519,5421200201,1968-11-30,736.0,True
962686,5421200201,1965-01-31,733.0,True
963335,5421200201,1968-04-30,673.0,True
962689,5421200201,1965-04-30,663.0,True
963520,5421200201,1968-12-31,650.0,True
963521,5421200201,1972-01-31,623.0,True
962444,5421200201,1962-04-30,615.0,True
962443,5421200201,1962-03-31,601.0,True
962688,5421200201,1965-03-31,588.0,True
962690,5421200201,1965-05-31,568.0,True


## 7. Ceros sospechosos y patrones de digitación

Un indicio de cero usado como faltante puede ser una estación-año con proporción de ceros casi total o bloques de fechas idénticos a cero. Es solo una bandera: una zona seca también puede registrar muchos días secos. Buscamos además valores extremos que concentren una gran parte del total anual observado.

In [15]:
zero_check = coverage.copy()
zero_check['zero_pct_rows'] = 100 * zero_check.zero_count / zero_check.rows
print('Estación-año con >= 95% filas cero (revisar, no borrar):', int((zero_check.zero_pct_rows >= 95).sum()))
display(zero_check.loc[zero_check.zero_pct_rows >= 95].sort_values('zero_pct_rows', ascending=False).head(20))

x = station.dropna(subset=['STATION_ID','DATE','VALUE_num']).copy()
x['year'] = x.DATE.dt.year
annual = x.groupby(['source','STATION_ID','year']).VALUE_num.sum().rename('annual_sum').reset_index()
x = x.merge(annual, on=['source','STATION_ID','year'], how='left')
x['daily_share_of_annual'] = np.where(x.annual_sum > 0, x.VALUE_num / x.annual_sum, np.nan)
suspect_days = x.loc[(x.VALUE_num >= 100) | (x.daily_share_of_annual >= .5),
                     ['source','STATION_ID','DATE','VALUE_num','annual_sum','daily_share_of_annual']]
display(suspect_days.sort_values('VALUE_num', ascending=False).head(30))

Estación-año con >= 95% filas cero (revisar, no borrar): 1


,source,STATION_ID,YEAR_INT,rows,unique_dates,zero_count,annual_sum_observed,first_date,last_date,expected_days,missing_calendar_days,coverage_pct,zero_pct_rows
7340,excel_1930_1979,2620000203,1979,1,1,1,0.0,1979-01-01,1979-01-01,365,364,0.273973,100.0


,source,STATION_ID,DATE,VALUE_num,annual_sum,daily_share_of_annual
343718,csv_1980_1990,2611600101,1984-01-10,859375.0,863235.0,0.995528
147876,csv_1980_1990,5421200201,1981-10-31,943.0,7713.0,0.122261
112351,csv_1980_1990,5421200201,1981-05-31,896.0,7713.0,0.116168
105202,csv_1980_1990,5421200201,1981-04-30,851.0,7713.0,0.110333
596988,csv_1980_1990,5300000201,1986-09-30,838.0,5837.0,0.143567
126496,csv_1980_1990,5421200201,1981-07-31,750.0,7713.0,0.097238
589279,csv_1980_1990,5300000201,1986-08-31,732.0,5837.0,0.125407
573342,csv_1980_1990,5300000201,1986-06-30,681.0,5837.0,0.116670
565607,csv_1980_1990,5300000201,1986-05-31,630.0,5837.0,0.107932
1960307,csv_1991_2003,2614400105,2003-04-09,600.0,2689.0,0.223131


## 8. Periodos por fuente y posibilidad de emparejamiento

Los CSV de estaciones cubren 1980–2003 en conjunto; el archivo histórico llega hasta 1979. El satélite comienza en 1981 y llega a 2013, aunque 2013 tiene solo 24 archivos. El periodo común no se debe fijar solo por extremos de fechas: también depende de cobertura estación-año y de tener ubicación para asignar el píxel.

In [18]:
periods = []
for name, df in list(frames.items()) + [('excel_1930_1979', hist)]:
    y = pd.to_numeric(df.YEAR, errors='coerce')
    periods.append({'fuente': name, 'filas': len(df), 'estaciones': df.STATION_ID.nunique(),
                    'primer_anio': int(y.min()), 'ultimo_anio': int(y.max())})
display(pd.DataFrame(periods))

all_ids = {name: set(df.STATION_ID.dropna().astype(str)) for name, df in list(frames.items()) + [('excel_1930_1979', hist)]}
print('IDs compartidos CSV 1980-90 y CSV 1991-2003:', len(all_ids['csv_1980_1990'] & all_ids['csv_1991_2003']))
print('Ojo: ID compartido no garantiza coordenadas iguales ni continuidad del sensor.')

,fuente,filas,estaciones,primer_anio,ultimo_anio
0,csv_1980_1990,997791,302,1980,1990
1,csv_1991_2003,1010677,284,1991,2003
2,excel_1930_1979,984226,264,1930,1979


IDs compartidos CSV 1980-90 y CSV 1991-2003: 261
Ojo: ID compartido no garantiza coordenadas iguales ni continuidad del sensor.


## 9. Inventario de imágenes satelitales

Los nombres `pptYYYYNN.tif` parecen codificar año y periodo dentro del año. El inventario verifica cuántos TIFF existen por año y omite los archivos `._...` que macOS agrega como metadatos auxiliares. Este conteo no confirma unidades, resolución, proyección, nodata ni si los valores son acumulados pentadales.

In [17]:
sat_zip = ROOT / 'ImagenesSatelitePentadal.zip'
with zipfile.ZipFile(sat_zip) as z:
    names = z.namelist()
    tiffs = [n for n in names if n.lower().endswith('.tif') and '/._' not in n]
    apple_meta = [n for n in names if '/._' in n]
    year_counts = Counter(Path(n).stem[3:7] for n in tiffs)
sat_year = pd.DataFrame([{'year': int(y), 'tiff_count': n, 'complete_72_expected': n == 72}
                         for y, n in sorted(year_counts.items())])
print('TIFF de datos:', len(tiffs), '| archivos AppleDouble:', len(apple_meta))
display(sat_year)

if PLOT_AVAILABLE:
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(sat_year.year, sat_year.tiff_count, color=np.where(sat_year.complete_72_expected, 'steelblue', 'darkorange'))
    ax.axhline(72, color='black', linestyle='--', linewidth=1, label='72 periodos/año según patrón')
    ax.set(title='Cantidad de TIFF por año', xlabel='Año', ylabel='Número de TIFF'); ax.legend(); ax.grid(axis='y', alpha=.2); plt.show()

# Intento opcional de leer metadatos de un TIFF. Si rasterio no está instalado, el conteo anterior sigue siendo válido.
try:
    import rasterio
    sample_path = '/tmp/eda_ppt_sample.tif'
    with zipfile.ZipFile(sat_zip) as z, open(sample_path, 'wb') as f:
        f.write(z.read(tiffs[0]))
    with rasterio.open(sample_path) as ds:
        print({'archivo': Path(tiffs[0]).name, 'ancho': ds.width, 'alto': ds.height,
               'bandas': ds.count, 'CRS': str(ds.crs), 'nodata': ds.nodata,
               'tipo': ds.dtypes, 'transform': ds.transform, 'bounds': ds.bounds})
        band = ds.read(1, masked=True)
        print('mínimo/máximo válidos:', float(band.min()), float(band.max()))
except ImportError:
    print('rasterio no está instalado: no se leyeron metadatos ni valores de píxel. Instálalo en tu entorno si el curso lo permite.')
except Exception as e:
    print('No se pudo leer el TIFF con rasterio:', repr(e))

TIFF de datos: 2328 | archivos AppleDouble: 2328


,year,tiff_count,complete_72_expected
0,1981,72,True
1,1982,72,True
2,1983,72,True
3,1984,72,True
4,1985,72,True
...,...,...,...
28,2009,72,True
29,2010,72,True
30,2011,72,True
31,2012,72,True


rasterio no está instalado: no se leyeron metadatos ni valores de píxel. Instálalo en tu entorno si el curso lo permite.


## 10. Conclusiones iniciales y decisiones pendientes

### Lo que este EDA sí establece
- Hay una brecha temporal entre la serie histórica de estaciones (1930–1979), los CSV (1980–2003) y el satélite (1981–2013).
- La gran proporción de ceros hace necesario validar códigos de calidad y registros faltantes antes de interpretar lluvia seca.
- La cobertura debe evaluarse por estación-año, no solo por cantidad total de filas.
- Los extremos y días que concentran gran parte del total anual requieren investigación contextual.
- El satélite parece tener 72 periodos anuales en 1981–2012 y solo 24 en 2013.

### Lo que falta antes de modelar
1. Catálogo de estaciones con latitud, longitud, altitud, nombre y cambios de sensor.
2. Diccionario de `VALUE`, unidad, códigos de faltantes y reglas de acumulación de lluvia.
3. Metadatos del TIFF: CRS, resolución espacial, unidades, escala, nodata y significado de los 72 periodos.
4. Regla reproducible para cobertura mínima y tratamiento de duplicados, acumulados y ceros sospechosos.
5. Confirmar cómo evaluar el periodo 2004–2013, donde este inventario no encuentra CSV de estaciones.

**No elimines ni imputes datos por una regla automática basada solo en este EDA.** Registra la decisión y compara cuántas estaciones y pares estación-año quedan antes y después de cada filtro.